In [914]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table

In [915]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [916]:
V1V3_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_df = biom_to_df(V1V3_path)
V1V3_df

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,...,g__Ramlibacter,g__Delftia,g__Lacibacter,g__Luteitalea,g__Pseudochrobactrum,g__Methylotenera,g__Fusicatenibacter,g__Ruminococcus,g__Lachnospira,g__Parabacteroides
LAMI.RD001.D0.C1,2362.0,0.0,94.0,292.0,175.0,91.0,33.0,37.0,18.0,1.0,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,1808.0,2.0,158.0,374.0,352.0,120.0,36.0,52.0,27.0,165.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C1,2234.0,2.0,83.0,240.0,492.0,253.0,19.0,30.0,30.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C2,1761.0,0.0,137.0,446.0,456.0,151.0,69.0,16.0,100.0,26.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D28.C1,2367.0,11.0,118.0,293.0,365.0,217.0,34.0,22.0,38.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,1900.0,1846.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C3,1003.0,2741.0,11.0,1.0,4.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,1410.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D9.C1,1230.0,2491.0,13.0,1.0,3.0,31.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [917]:
wgs_pathV4_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_df = biom_to_df(V4_path)
V4_df

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Anaerostipes,g__Dietzia,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella
LAMI.RD001.D0.C1,317.0,61.0,131.0,275.0,414.0,116.0,475.0,100.0,15.0,706.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0
LAMI.RD001.D14.C1,454.0,29.0,456.0,915.0,247.0,57.0,168.0,93.0,117.0,214.0,...,3.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD004.D0.C2,1652.0,36.0,95.0,848.0,164.0,72.0,28.0,81.0,97.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,349.0,46.0,217.0,509.0,368.0,83.0,464.0,75.0,182.0,543.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,0.0,0.0
LAMI.RD004.D28.C1,332.0,16.0,54.0,964.0,422.0,326.0,31.0,118.0,27.0,4.0,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D16.C2,142.0,3413.0,57.0,48.0,2.0,3.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D28.C1,42.0,3525.0,96.0,38.0,7.0,5.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD318.D9.C2,139.0,3441.0,23.0,27.0,11.0,35.0,2.0,12.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D28.C2,52.0,3627.0,54.0,15.0,1.0,0.0,1.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [918]:
wgs_path = '../Data/metaG/Tables/VEBA_MAGs_collapsed_genus_absolute.biom'
# wgs_path = '../Data/metaG/Tables/metaG_rs210_micov-filt_genus.biom'
wgs_df = biom_to_df(wgs_path).T
wgs_df.index = wgs_df.index.str.replace('_', '.', regex=False)
wgs_df.index = wgs_df.index.to_series().str.split('.').str[:4].str.join('.')

wgs_df = wgs_df.rename(columns={'Corynebacterium spp.': ' g__Corynebacterium'})
wgs_df

,Caudoviricetes,g__Corynebacterium,Cutibacterium acnes,Cutibacterium granulosum,Lawsonella cleavelandensis,Malassezia spp.,Micrococcus spp.,Neisseria spp.,Other (combined),Papillomaviridae,Streptococcus mitis
LAMI.RD304.D28.C3,222.0,93790.0,20742697.0,191387.0,27869.0,297999.0,36407.0,16560.0,45682.0,61.0,1495.0
LAMI.RD306.D14.C3,59845.0,206439.0,13837161.0,553526.0,1331392.0,1262341.0,42540.0,58340.0,98313.0,44785.0,7516.0
LAMI.RD306.D28.C3,56329.0,334721.0,13562865.0,1175074.0,286805.0,1015542.0,151610.0,37267.0,132301.0,151848.0,5756.0
LAMI.RD306.D23.C1,11514.0,245579.0,15580453.0,76875.0,85938.0,84174.0,7976.0,89126.0,14262.0,3520.0,632.0
LAMI.RD308.D9.C3,370.0,58549.0,14785186.0,61142.0,15843.0,84768.0,58697.0,10580.0,69774.0,18.0,11842.0
LAMI.RD308.D11.C2,197.0,47274.0,13642581.0,37069.0,20277.0,75190.0,28873.0,4563.0,20414.0,0.0,4303.0
LAMI.RD308.D14.C3,379.0,32145.0,12966108.0,39956.0,13269.0,26343.0,9357.0,1146.0,8176.0,0.0,534.0
LAMI.RD310.D0.C3,2708.0,98796.0,11499482.0,70100.0,10053.0,195785.0,14322.0,2050.0,23779.0,12646.0,130.0
LAMI.RD310.D21.C2,5237.0,94854.0,10625365.0,73714.0,9723.0,166989.0,7483.0,4030.0,40009.0,8.0,425.0
LAMI.RD306.D11.C1,136813.0,104299.0,7731248.0,424393.0,366520.0,518869.0,52410.0,21673.0,1624688.0,6829.0,6435.0


In [919]:
# Import in the metadata
metadata = pd.read_csv('../Metadata/metadata_df.csv')
metadata.columns
metadata = metadata[['SampleID', 'local_lesion_severity', 'host_subject_id', 'day']]
metadata = metadata.set_index('SampleID')
metadata.index.name = None

indexes_to_include = V4_df.index.to_list()

metadata = metadata.loc[metadata.index.isin(indexes_to_include)]
metadata

,local_lesion_severity,host_subject_id,day
LAMI.RD308.D16.C1,4,RD308,16
LAMI.RD310.D21.C1,4,RD310,21
LAMI.RD305.D21.C3,0,RD305,21
LAMI.RD306.D18.C2,2,RD306,18
LAMI.RD306.D7.C2,3,RD306,7
...,...,...,...
LAMI.RD305.D0.C2,3,RD305,0
LAMI.RD317.D21.C1,1,RD317,21
LAMI.RD001.D0.C1,0,RD001,0
LAMI.RD314.D0.C1,1,RD314,0


In [920]:
md_RD304 = metadata[metadata['host_subject_id'] == 'RD304']
md_RD304_C1 = md_RD304[md_RD304.index.str.endswith('C1')]
md_RD304_C1_samples = md_RD304_C1.index.to_list()
md_RD304_C1

,local_lesion_severity,host_subject_id,day
LAMI.RD304.D7.C1,4,RD304,7
LAMI.RD304.D11.C1,2,RD304,11
LAMI.RD304.D28.C1,3,RD304,28
LAMI.RD304.D4.C1,3,RD304,4
LAMI.RD304.D16.C1,4,RD304,16
LAMI.RD304.D0.C1,3,RD304,0
LAMI.RD304.D14.C1,3,RD304,14
LAMI.RD304.D9.C1,3,RD304,9


In [921]:
# Subset data tables to be same samples
V1V3_df_RD304_C1 = V1V3_df.loc[V1V3_df.index.isin(md_RD304_C1_samples)]
V4_df_RD304_C1 = V4_df.loc[V4_df.index.isin(md_RD304_C1_samples)]
wgs_df_RD304_C1 = wgs_df.loc[wgs_df.index.isin(md_RD304_C1_samples)]

V1V3_df_RD304_C1

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,...,g__Ramlibacter,g__Delftia,g__Lacibacter,g__Luteitalea,g__Pseudochrobactrum,g__Methylotenera,g__Fusicatenibacter,g__Ruminococcus,g__Lachnospira,g__Parabacteroides
LAMI.RD304.D16.C1,3416.0,0.0,122.0,15.0,109.0,8.0,1.0,1.0,2.0,91.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D7.C1,3377.0,2.0,228.0,15.0,64.0,14.0,3.0,2.0,0.0,57.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D0.C1,3409.0,1.0,268.0,18.0,40.0,7.0,6.0,6.0,0.0,10.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D9.C1,2975.0,1.0,717.0,7.0,28.0,7.0,0.0,2.0,0.0,26.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D11.C1,3509.0,2.0,117.0,63.0,18.0,1.0,7.0,3.0,4.0,5.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D4.C1,3552.0,2.0,131.0,6.0,28.0,7.0,0.0,3.0,0.0,34.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D14.C1,3640.0,5.0,75.0,25.0,10.0,2.0,2.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [922]:
V4_df_RD304_C1

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Anaerostipes,g__Dietzia,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella
LAMI.RD304.D0.C1,2249.0,27.0,22.0,268.0,105.0,490.0,9.0,26.0,65.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D9.C1,2977.0,19.0,48.0,206.0,24.0,35.0,5.0,21.0,136.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D11.C1,1766.0,57.0,48.0,199.0,487.0,98.0,206.0,47.0,67.0,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D14.C1,2198.0,15.0,63.0,281.0,367.0,49.0,68.0,16.0,117.0,13.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D28.C1,1964.0,29.0,63.0,303.0,141.0,44.0,20.0,37.0,608.0,30.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D16.C1,1460.0,33.0,86.0,824.0,94.0,34.0,34.0,19.0,642.0,7.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D4.C1,1901.0,51.0,88.0,641.0,58.0,127.0,40.0,58.0,248.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD304.D7.C1,2238.0,20.0,103.0,386.0,67.0,85.0,20.0,53.0,335.0,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [923]:
wgs_df_RD304_C1

,Caudoviricetes,g__Corynebacterium,Cutibacterium acnes,Cutibacterium granulosum,Lawsonella cleavelandensis,Malassezia spp.,Micrococcus spp.,Neisseria spp.,Other (combined),Papillomaviridae,Streptococcus mitis
LAMI.RD304.D7.C1,109.0,68121.0,7913128.0,83954.0,22420.0,537906.0,32463.0,7749.0,26082.0,3422.0,169.0
LAMI.RD304.D0.C1,165.0,46794.0,6320982.0,77122.0,12237.0,720488.0,24638.0,8428.0,23346.0,264.0,626.0
LAMI.RD304.D11.C1,38.0,1959.0,327651.0,2620.0,1295.0,10294.0,899.0,509.0,1614.0,3.0,243.0


In [924]:
# Add pseudocount
V1V3_df_RD304_C1_pseudo = V1V3_df_RD304_C1 + 1

# CLR transformation function
def clr_transform(df):
    log_df = np.log(df)
    geometric_mean = log_df.mean(axis=1)
    clr_df = log_df.subtract(geometric_mean, axis=0)
    return clr_df

# Apply CLR
V1V3_df_RD304_C1_clr = clr_transform(V1V3_df_RD304_C1_pseudo)
V1V3_df_RD304_C1_clr

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,...,g__Ramlibacter,g__Delftia,g__Lacibacter,g__Luteitalea,g__Pseudochrobactrum,g__Methylotenera,g__Fusicatenibacter,g__Ruminococcus,g__Lachnospira,g__Parabacteroides
LAMI.RD304.D16.C1,7.644265,-0.492254,4.319931,2.280335,4.208227,1.704971,0.200893,0.200893,0.606359,4.029535,...,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254,-0.492254
LAMI.RD304.D7.C1,7.573786,0.547359,4.882469,2.221336,3.623134,2.156797,0.835041,0.547359,-0.551253,3.509190,...,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253,-0.551253
LAMI.RD304.D0.C1,7.623006,0.181686,5.083250,2.432977,3.202110,1.567980,1.434449,1.434449,-0.511462,1.886434,...,0.181686,-0.511462,-0.511462,-0.511462,-0.511462,-0.511462,-0.511462,-0.511462,-0.511462,-0.511462
LAMI.RD304.D9.C1,7.522761,0.217573,6.100895,1.603867,2.891722,1.603867,-0.475574,0.623038,-0.475574,2.820263,...,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574,-0.475574
LAMI.RD304.D11.C1,7.508794,0.444035,4.116107,3.504306,2.289861,0.038570,1.424864,0.731717,0.954860,1.137182,...,-0.654578,0.038570,-0.654578,-0.654578,-0.654578,-0.654578,-0.654578,-0.654578,-0.654578,-0.654578
LAMI.RD304.D4.C1,7.693604,0.616669,4.400858,1.463966,2.885352,1.597498,-0.481944,0.904351,-0.481944,3.073404,...,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944,-0.481944
LAMI.RD304.D14.C1,7.783028,1.374774,3.913748,2.841111,1.980910,0.681627,0.681627,-0.416985,-0.416985,0.969309,...,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985,-0.416985


In [925]:
# Add pseudocount
V4_df_RD304_C1_pseudo = V4_df_RD304_C1 + 1

# CLR transformation function
def clr_transform(df):
    log_df = np.log(df)
    geometric_mean = log_df.mean(axis=1)
    clr_df = log_df.subtract(geometric_mean, axis=0)
    return clr_df

# Apply CLR
V4_df_RD304_C1_clr = clr_transform(V4_df_RD304_C1_pseudo)
V4_df_RD304_C1_clr

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Neisseria,...,g__Anaerostipes,g__Dietzia,g__Conchiformibius,g__Treponema,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella
LAMI.RD304.D0.C1,6.514353,2.127872,1.931162,4.390379,3.459106,4.992112,1.098252,2.091504,2.985322,0.875109,...,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333,-1.204333
LAMI.RD304.D9.C1,6.963043,1.959768,2.855856,4.296755,2.182912,2.547555,0.755796,2.055079,3.884017,1.161261,...,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964,-1.035964
LAMI.RD304.D11.C1,6.108607,2.692012,2.523389,3.929886,4.821884,3.226689,3.964288,2.502770,2.851077,3.266298,...,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431,-1.368431
LAMI.RD304.D14.C1,6.439327,1.516158,2.902452,4.385476,4.651652,2.655592,2.977676,1.576782,3.514254,1.382626,...,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431,-1.256431
LAMI.RD304.D28.C1,6.253731,2.071680,2.829366,4.387511,3.626310,2.477145,1.715005,2.308069,5.082301,2.104470,...,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517,-1.329517
LAMI.RD304.D16.C1,5.947723,2.187208,3.126755,5.376230,3.214724,2.216195,2.216195,1.656579,5.126992,0.740289,...,-1.339153,-1.339153,-1.339153,-0.646006,-1.339153,-1.339153,-1.339153,-1.339153,-1.339153,-1.339153
LAMI.RD304.D4.C1,6.338085,2.738668,3.276061,5.252012,2.864962,3.639454,2.500996,2.864962,4.304877,-1.212576,...,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576,-1.212576
LAMI.RD304.D7.C1,6.373765,1.704503,3.304371,4.618405,2.879488,3.114328,1.704503,2.648965,4.477092,1.795475,...,-1.340020,-1.340020,-1.340020,-1.340020,-1.340020,-1.340020,-0.646872,-1.340020,-1.340020,-1.340020


In [926]:
# Add pseudocount
wgs_df_RD304_C1_pseudo = wgs_df_RD304_C1 + 1

# CLR transformation function
def clr_transform(df):
    log_df = np.log(df)
    geometric_mean = log_df.mean(axis=1)
    clr_df = log_df.subtract(geometric_mean, axis=0)
    return clr_df

# Apply CLR
wgs_df_RD304_C1_clr = clr_transform(wgs_df_RD304_C1_pseudo)
wgs_df_RD304_C1_clr

,Caudoviricetes,g__Corynebacterium,Cutibacterium acnes,Cutibacterium granulosum,Lawsonella cleavelandensis,Malassezia spp.,Micrococcus spp.,Neisseria spp.,Other (combined),Papillomaviridae,Streptococcus mitis
LAMI.RD304.D7.C1,-5.213269,1.215306,5.970284,1.424287,0.104004,3.281691,0.474137,-0.958301,0.255289,-1.775477,-4.777951
LAMI.RD304.D0.C1,-4.606964,1.034580,5.940434,1.534205,-0.306650,3.768734,0.393134,-0.679518,0.339272,-4.139222,-3.278005
LAMI.RD304.D11.C1,-3.202719,0.714419,5.833426,1.005030,0.300757,2.373133,-0.063886,-0.631870,0.520809,-5.479987,-1.369113


In [927]:
md_RD304_C1['g__Corynebacterium V1V3'] = V1V3_df_RD304_C1_clr[' g__Corynebacterium']
md_RD304_C1['g__Corynebacterium V4'] = V4_df_RD304_C1_clr[' g__Corynebacterium']
md_RD304_C1['g__Corynebacterium WGS'] = wgs_df_RD304_C1_clr[' g__Corynebacterium']

md_RD304_C1 = md_RD304_C1.dropna(subset=['g__Corynebacterium V1V3', 'g__Corynebacterium V4'])

md_RD304_C1

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_35140/511337277.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  md_RD304_C1['g__Corynebacterium V1V3'] = V1V3_df_RD304_C1_clr[' g__Corynebacterium']
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_35140/511337277.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  md_RD304_C1['g__Corynebacterium V4'] = V4_df_RD304_C1_clr[' g__Corynebacterium']
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_35140/511337277.py:3: SettingWith

,local_lesion_severity,host_subject_id,day,g__Corynebacterium V1V3,g__Corynebacterium V4,g__Corynebacterium WGS
LAMI.RD304.D7.C1,4,RD304,7,3.623134,4.618405,1.215306
LAMI.RD304.D11.C1,2,RD304,11,2.289861,3.929886,0.714419
LAMI.RD304.D4.C1,3,RD304,4,2.885352,5.252012,NaN
LAMI.RD304.D16.C1,4,RD304,16,4.208227,5.376230,NaN
LAMI.RD304.D0.C1,3,RD304,0,3.202110,4.390379,1.034580
LAMI.RD304.D14.C1,3,RD304,14,1.980910,4.385476,NaN
LAMI.RD304.D9.C1,3,RD304,9,2.891722,4.296755,NaN


## Plot correlation figures

In [929]:
# Sort by day for better plotting
df = md_RD304_C1.sort_values('day')

df_wgs = df.dropna(subset=['g__Corynebacterium WGS', 'day'])

# Correlation: V1V3
r1, p1 = pearsonr(df['local_lesion_severity'], df['g__Corynebacterium V1V3'])

# Correlation: V4
r2, p2 = pearsonr(df['local_lesion_severity'], df['g__Corynebacterium V4'])

# Correlation: WGS — use dropna to avoid errors
df_clean = df_wgs[['local_lesion_severity', 'g__Corynebacterium WGS']].dropna()
r3, p3 = pearsonr(df_clean['local_lesion_severity'], df_clean['g__Corynebacterium WGS'])

# Start figure
fig, ax1 = plt.subplots(figsize=(6, 6))

# Plot local lesion severity
ax1.plot(df['day'], df['local_lesion_severity'], color='red', marker='o', label='Acne Severity')
ax1.set_ylabel('Acne Severity (Subject 304)', color='red', fontsize=12)
ax1.tick_params(axis='y', labelcolor='black')
ax1.set_ylim(0, 5.5)

# Secondary y-axis for Corynebacterium CLR
ax2 = ax1.twinx()

# V1V3 (light blue squares)
ax2.plot(df['day'], df['g__Corynebacterium V1V3'], color='#86ceeb', marker='s', label='Corynebacterium (V1V3)')

# V4 (purple diamonds)
ax2.plot(df['day'], df['g__Corynebacterium V4'], color='#dda0dd', marker='D', label='Corynebacterium (V4)')

# WGS (orange triangles)
ax2.plot(
    df_wgs['day'], 
    df_wgs['g__Corynebacterium WGS'], 
    color='orange', marker='^', label='Corynebacterium (WGS)'
)

ax2.set_ylabel('Corynebacterium (CLR)', color='black', fontsize=12)
ax2.tick_params(axis='y', labelcolor='black')
ax2.set_ylim(0, 5.5)

# Title and labels
# plt.title("Subject 304: Acne Severity vs. Corynebacterium Abundance", fontsize=13)
ax1.set_xlabel('Day', color='black', fontsize=12)

# Annotate correlations
ax1.text(0.48, 0.95, f"V1V3: r={r1:.2f}, p={'< 0.001' if p1 < 0.001 else f'{p1:.3f}'}", transform=ax1.transAxes, fontsize=10)
ax1.text(0.48, 0.90, f"V4: r={r2:.2f}, p={'< 0.001' if p2 < 0.001 else f'{p2:.3f}'}", transform=ax1.transAxes, fontsize=10)
ax1.text(0.48, 0.85, f"WGS: r={r3:.2f}, p={'< 0.001' if p3 < 0.001 else f'{p3:.3f}'}", transform=ax1.transAxes, fontsize=10)

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax2.get_legend_handles_labels()

ax1.legend(lines_1 + lines_2, labels_1 + labels_2, 
           loc='upper center', bbox_to_anchor=(0.5, -0.15), 
           ncol=2, fontsize=10, frameon=True)

# Tidy layout and save
plt.tight_layout()
plt.savefig('../Figures/multi-omics_Figures/subj_304_C1_corynebacterium_V1V3_V4_WGS.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_35140/3167531311.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
